# S163 — ARGOS image sanitizer MVP (LaMa via IOPaint)

**Scopo**: GO/NO-GO sulla qualita' di LaMa per rimuovere watermark + targhe dalle foto auto dealer.

**Setup**: Colab T4 free (Runtime -> Change runtime type -> T4 GPU).

**Cosa fa**: avvia il server IOPaint con modello LaMa (auto-download ~200MB al primo run) e lo espone via ngrok. Tu apri l'URL pubblico, carichi 1 foto alla volta, disegni la mask con il pennello sulle aree da rimuovere, click `Erase` e scarichi il risultato.

**Prerequisito**: NGROK_AUTHTOKEN nel pannello Colab `Secrets` (icona chiave a sinistra). Token gratuito su https://dashboard.ngrok.com/get-started/your-authtoken (gia' presente in `~/.claude/.env.free-gpu` su MacBook).

**Budget tempo**: 10min boot + ~200MB download LaMa + 3-5min/foto annotazione manuale = ~30-45min totali per 3 foto sample.

**Vincoli applicati**: #1 verifica fattuale (iopaint 1.6.0 verificato PyPI marzo 2025), #5 zero-cost (Colab T4 + IOPaint MIT + ngrok free), #10 verificato vs verosimile (no pipeline custom, solo wrapper di app esistente).

## 1. Verifica GPU disponibile

In [ ]:
!nvidia-smi

## 2. Install IOPaint + pyngrok

~3-5min. IOPaint installa torch, fastapi, uvicorn, pillow, opencv-python e tutte le dipendenze necessarie.

In [ ]:
!pip install -q iopaint==1.6.0 pyngrok

## 3. Carica ngrok authtoken dai Secrets Colab

Apri il pannello Secrets (icona chiave a sinistra) e aggiungi:
- Name: `NGROK_AUTHTOKEN`
- Value: il tuo token (gia' in `~/.claude/.env.free-gpu` su MacBook)
- Toggle Notebook access ON

In [ ]:
from google.colab import userdata
import os
os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')
print('ngrok token loaded:', len(os.environ['NGROK_AUTHTOKEN']) > 0)

## 4. Avvia IOPaint server in background

Lancia il server FastAPI + web UI di IOPaint sulla porta 8080. Al primo avvio scarica il checkpoint LaMa (~200MB) da Hugging Face. Aspetta ~30-60s che il log mostri `Uvicorn running on http://0.0.0.0:8080`.

In [ ]:
import subprocess, time, os

log_file = open('/content/iopaint.log', 'w')
proc = subprocess.Popen(
    ['iopaint', 'start', '--model=lama', '--device=cuda', '--port=8080', '--host=0.0.0.0'],
    stdout=log_file, stderr=subprocess.STDOUT,
)
print(f'IOPaint PID: {proc.pid}')
print('Waiting for server to be ready (download LaMa ~200MB)...')

ready = False
for i in range(180):  # max 3min
    time.sleep(2)
    with open('/content/iopaint.log') as f:
        log = f.read()
    if 'Uvicorn running on' in log:
        ready = True
        print(f'\nServer ready after {i*2}s')
        break
    if i % 5 == 0:
        print(f'  ...{i*2}s', end=' ')

if not ready:
    print('\nTIMEOUT — controlla /content/iopaint.log')
    !tail -40 /content/iopaint.log

## 5. Apri tunnel ngrok e stampa URL pubblico

In [ ]:
from pyngrok import conf, ngrok

conf.get_default().auth_token = os.environ['NGROK_AUTHTOKEN']
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
tunnel = ngrok.connect(8080, 'http')
print(f'\n{"="*60}\n  IOPAINT WEB UI: {tunnel.public_url}\n{"="*60}\n')
print('Apri questo URL nel browser. Carica 1 foto, disegna mask con pennello\nsui watermark/targhe, click Erase, scarica risultato.')

## 6. Workflow MVP — 3 foto sample

1. **Foto A** (watermark dealer): apri URL ngrok, carica foto, pennello su logo/testo, Erase, scarica come `sanitized_A.png`
2. **Foto B** (targa visibile): pennello sulla targa, Erase, scarica come `sanitized_B.png`
3. **Foto C** (entrambi): mask multipla in un solo pass, Erase, scarica come `sanitized_C.png`

**Criteri GO/NO-GO** (decisione S163):
- GO: 3/3 risultati con area inpaint plausibile, no artefatti grossolani visibili a 100% zoom
- NO-GO: artefatti visibili (banding, halo, texture ripetuta) -> upgrade S163-bis a BrushNet/PowerPaint v2 (stesso IOPaint platform, switch `--model=brushnet` o `--model=powerpaint`)

**Salva 3 originali + 3 sanitized** in `~/Documents/combaretrovamiauto-enterprise/sanitizer_colab/outputs/` per audit retroattivo.

## 7. Cleanup (opzionale, alla fine sessione)

In [ ]:
# ngrok.disconnect(tunnel.public_url)
# proc.terminate()
# print('Tunnel chiuso, server fermato.')